# Create or delete a Weaviate collection

In [ ]:
import os
import logging

from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.config import AdditionalConfig, Timeout
from weaviate.classes.config import Configure

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

## Delete a collection
🚨 WARNING: DO NOT RUN THIS UNLESS YOU WANT TO DELETE A COLLECTION 🚨

Deletes a collection and all of its entries.

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Default is 80, WCD uses 443
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,  # Default is 50051, WCD uses 443
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),  # The API key to use for authentication
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=60, insert=120)  # Values in seconds
        )
    )
    INDEX_NAME = "Ingestion_20250609"
    collection_name = client.collections.get(INDEX_NAME)
    client.collections.delete(INDEX_NAME)

except Exception as e:
    print(e)
finally:
    client.close()

## Create
Create a new Weaviate collection. There are two important factors to consider when making a collection:
1. Whether to specify a vectorizer or not. Either way is fine, but if you specify a vectorizer, this will allow you to use LangChain functions like `add_documents()`. Otherwise, you will have to be more creative in how you embed and add documents to your collection. I find it best to specify a vectorizer, since changing the embedding model means ingesting from scratch anyway.
2. Whether to explicitly define properties or not. Leaving properties blank will allow Weaviate's auto-schema to create them for you, which I have found works well. In the future though, we may want to better standardize the metadata across sources ( like creation date), in which case writing out the properties here could allow us to be more specific.

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=60, insert=120)
        )
    )
    INDEX_NAME = "test_20250612"
    client.collections.create(
        INDEX_NAME,
        vectorizer_config=[
            Configure.NamedVectors.text2vec_openai(
                name="default",
                model="text-embedding-3-small",
                dimensions=1536,
            )
        ]
        # No 'properties' argument: enables auto-schema
    )

except Exception as e:
    print(e)
finally:
    client.close()